# Algorithme de Christofides — TSP

L'algorithme de Christofides est une heuristique pour le **Problème du Voyageur de Commerce (TSP)** qui garantit une solution dont le coût ne dépasse pas **1.5× l'optimal**, à condition que le graphe respecte l'inégalité triangulaire (vrai pour les distances euclidiennes).

### Les 5 étapes

| # | Étape | Description |
|---|-------|-------------|
| 1 | **MST** | Arbre couvrant minimal sur le graphe complet |
| 2 | **Sommets impairs** | Sommets du MST ayant un degré impair |
| 3 | **Matching** | Couplage parfait de poids minimum sur ces sommets |
| 4 | **Multigraphe** | Fusion MST + matching → tous les degrés deviennent pairs |
| 5 | **Circuit** | Circuit eulérien puis court-circuitage → circuit hamiltonien |

## 0. Imports

In [ ]:
import math
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

## 1. Génération du graphe

On génère un **graphe complet euclidien** : `n` points aléatoires en 2D, reliés entre eux par leur distance euclidienne. L'inégalité triangulaire est automatiquement respectée, ce qui est la condition requise pour Christofides.

In [ ]:
def generate_graph_from_json(path):
    with open(path, "r") as f:
        data = json.load(f)

    # Récupération des noeuds
    nodes = [data["depot"]] + data["clients"]

    G = nx.Graph()

    # Ajouter les noeuds avec positions
    for i, node in enumerate(nodes):
        G.add_node(i, pos=(node["x"], node["y"]))

    # Graphe complet avec distances euclidiennes
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            xi, yi = nodes[i]["x"], nodes[i]["y"]
            xj, yj = nodes[j]["x"], nodes[j]["y"]
            dist = math.hypot(xi - xj, yi - yj)
            G.add_edge(i, j, weight=dist)

    return G, data


def draw(G, edge_groups=None, title="Graphe", grey_edges=True, node_style="big"):
    """
    Affiche le graphe G en dessinant les arêtes en deux passes :
    d'abord les grises (fond), puis les colorées par-dessus.
    edge_groups : liste de (edges, couleur, épaisseur)
    """
    pos = nx.get_node_attributes(G, "pos")

    highlighted = {}
    if edge_groups:
        for edges, color, width in edge_groups:
            for u, v in edges:
                highlighted[(min(u,v), max(u,v))] = (color, width)

    gray_edges = [(u, v) for u, v in G.edges() if (min(u,v), max(u,v)) not in highlighted]
    
    fig, ax = plt.subplots(figsize=(8, 6))

    # Passe 1 — arêtes grises en fond
    if grey_edges:
        nx.draw_networkx_edges(G, pos, edgelist=gray_edges, edge_color="lightgray", width=1, ax=ax)

    # Passe 2 — arêtes colorées par-dessus, groupe par groupe
    if edge_groups:
        for edges, color, width in edge_groups:
            nx.draw_networkx_edges(G, pos, edgelist=edges, edge_color=color, width=width, ax=ax)

    # Noeuds et labels
    if node_style == "big":
        nx.draw_networkx_nodes(G, pos, node_color="steelblue", node_size=500, ax=ax)
        nx.draw_networkx_labels(G, pos, font_color="white", ax=ax)
    else:
        nx.draw_networkx_nodes(G, pos, node_color="steelblue", node_size=50, ax=ax)

    ax.set_title(title)
    plt.tight_layout()
    plt.show()


G, data = generate_graph_from_json("datasets/tsptwd_n10.json")
draw(G, title="Graphe complet euclidien (10 sommets)")

## 2. Arbre Couvrant Minimal (MST) — Algorithme de Prim

On construit un MST avec l'algorithme de **Prim** :
- On maintient un tableau `key[v]` = coût minimum pour rattacher `v` à l'arbre
- À chaque étape, on ajoute le sommet non visité avec le `key` le plus faible
- On met à jour les `key` des voisins si on trouve une arête moins chère

Le MST retourne `n-1` arêtes qui connectent tous les sommets sans cycle.

In [ ]:
def prim(graph):
    n = len(graph)
    key = [math.inf] * n
    key[0] = 0
    parent = [None] * n
    visited = set()

    while len(visited) != n:
        # Sommet non visité avec le coût d'accès minimum
        not_visited = [v for v in range(n) if v not in visited]
        u = min(not_visited, key=lambda v: key[v])
        visited.add(u)

        # Mise à jour des voisins
        for v in graph[u]:
            if v not in visited:
                w = graph[u][v]['weight']
                if w < key[v]:
                    key[v] = w
                    parent[v] = u

    return [(parent[v], v) for v in range(n) if parent[v] is not None]


mst_edges = prim(G)
draw(G, edge_groups=[(mst_edges, "violet", 3)], title="Étape 1 — MST (Prim)")

## 3. Sommets de degré impair

Par le théorème des poignées de mains, il y a toujours un nombre **pair** de sommets de degré impair dans un graphe. On les identifie pour l'étape suivante.

In [ ]:
def odd_degree_nodes(mst_edges):
    cnt = Counter(v for edge in mst_edges for v in edge)
    return sorted(v for v in cnt if cnt[v] % 2 != 0)


odd_nodes = odd_degree_nodes(mst_edges)
print(f"Sommets de degré impair : {odd_nodes}")

## 4. Couplage parfait de poids minimum

On extrait le **sous-graphe induit** par les sommets impairs, puis on trouve le couplage parfait de poids minimum via l'algorithme de Blossom d'Edmonds (fourni par NetworkX).

Chaque sommet impair sera apparié exactement une fois, ce qui augmentera son degré de 1 → le rendra pair.

In [ ]:
def min_matching(G, odd_nodes):
    subgraph = G.subgraph(odd_nodes)
    return list(nx.min_weight_matching(subgraph))


matching_edges = min_matching(G, odd_nodes)
draw(G,
     edge_groups=[(mst_edges, "violet", 3), (matching_edges, "orange", 3)],
     title="Étape 1+2 — MST (violet) + Matching (orange)")

## 5. Multigraphe eulérien

On fusionne les arêtes du MST et du matching dans un `MultiGraph` (qui accepte plusieurs arêtes entre deux mêmes sommets). Après la fusion, **tous les sommets ont un degré pair** — condition nécessaire et suffisante pour l'existence d'un circuit eulérien (théorème d'Euler).

In [ ]:
def build_multigraph(G, mst_edges, matching_edges):
    MG = nx.MultiGraph()
    MG.add_nodes_from(G.nodes(data=True))
    MG.add_edges_from((u, v, {'weight': G[u][v]['weight']}) for u, v in mst_edges)
    MG.add_edges_from((u, v, {'weight': G[u][v]['weight']}) for u, v in matching_edges)
    return MG


multigraph = build_multigraph(G, mst_edges, matching_edges)
all_mg_edges = list(multigraph.edges())
draw(G,
     edge_groups=[(all_mg_edges, "steelblue", 2)],
     title="Étape 3 — Multigraphe eulérien")

## 6. Circuit eulérien → Circuit hamiltonien

1. On extrait un **circuit eulérien** (passe par chaque arête exactement une fois) via l'algorithme de Hierholzer.
2. On **court-circuite** les sommets déjà visités pour obtenir un circuit hamiltonien (chaque sommet exactement une fois). Cela est valide grâce à l'inégalité triangulaire : sauter une ville ne peut qu'améliorer ou maintenir le coût.

In [ ]:
def eulerian_to_hamiltonian(multigraph):
    visited = set()
    hamiltonian = []
    for u, v in nx.eulerian_circuit(multigraph):
        if u not in visited:
            hamiltonian.append(u)
            visited.add(u)
    hamiltonian.append(hamiltonian[0])  # Fermer le circuit
    return hamiltonian


hamiltonian = eulerian_to_hamiltonian(multigraph)

circuit = list(nx.eulerian_circuit(multigraph))
euler = [u for u, v in circuit] + [circuit[-1][1]]
print("Circuit eulérien  :", " → ".join(str(v) for v in euler))
print("Circuit hamiltonien:", " → ".join(str(v) for v in hamiltonian))

## 7. Algorithme complet

On regroupe toutes les étapes dans une seule fonction `christofides(G)`.

In [ ]:
def christofides(G):
    mst_edges      = prim(G)
    odd_nodes      = odd_degree_nodes(mst_edges)
    matching_edges = min_matching(G, odd_nodes)
    multigraph     = build_multigraph(G, mst_edges, matching_edges)
    hamiltonian    = eulerian_to_hamiltonian(multigraph)
    return hamiltonian


def tour_cost(G, hamiltonian, scale=200):
    return round(sum(G[hamiltonian[i]][hamiltonian[i+1]]['weight'] for i in range(len(hamiltonian) - 1)) * scale, 1)

# Résultat
G, _ = generate_graph_from_json("datasets/tsptwd_n10.json")
hamiltonian = christofides(G)
cost = tour_cost(G, hamiltonian)

ham_edges = list(zip(hamiltonian[:-1], hamiltonian[1:]))
draw(G,
     edge_groups=[(ham_edges, "green", 3)],
     title=f"Résultat Christofides — Coût : {cost:.4f}")
print(f"Circuit : {hamiltonian}")
print(f"Coût total : {cost:.4f}")

In [ ]:
def all_in_one(G):
    hamiltonian = christofides(G)
    cost = tour_cost(G, hamiltonian)

    ham_edges = list(zip(hamiltonian[:-1], hamiltonian[1:]))
    draw(G,
        edge_groups=[(ham_edges, "green", 3)],
        title=f"Résultat Christofides — Coût : {cost:.2f}",
        grey_edges=False,
        node_style="little")
    print(f"Circuit : {hamiltonian}")
    print(f"Coût total : {cost:.2f}")

G, _ = generate_graph_from_json("datasets/tsptwd_n100.json")
all_in_one(G)

## Benchmark

---

## TSPTW-D : Adaptation aux contraintes du Livrable 1

Le Christofides standard minimise la **somme des distances**. Avec les contraintes du TSPTW-D :

| Contrainte | Modèle retenu | Implémentation actuelle |
|---|---|---|
| Fenêtres temporelles | **Glissantes** multi-jour | ✗ ignorées |
| Temps de service | $D_i = \tau_i + w_i + s_i$ | ✗ ignoré |
| Coûts dynamiques | $c_{ij}(D_i) = c_{ij}^{\text{base}} \cdot \alpha$ si perturbation | ✗ ignoré |
| Coût évalué à | Heure de **départ** $D_i$ | ✗ (distances statiques) |

**Fenêtres glissantes (multi-jour) :** si le véhicule arrive hors fenêtre, il attend l'ouverture suivante. La tournée est donc **toujours faisable**, mais les arrivées tardives ajoutent du temps d'attente.

$$w_i = \begin{cases} a_i - \tau_i & \text{si } \tau_i < a_i \\ 0 & \text{si } a_i \leq \tau_i \leq b_i \\ a_i + k \cdot 1440 - \tau_i & \text{si } \tau_i > b_i,\quad k = \lceil(\tau_i - a_i)/1440\rceil \end{cases}$$

**Propagation temporelle :**
$$D_i = \tau_i + w_i + s_i \qquad \tau_{k+1} = D_{\sigma_k} + c_{\sigma_k,\sigma_{k+1}}(D_{\sigma_k})$$

**Adaptation :**
1. Le graphe est construit avec les distances de base (inchangé)
2. `tour_cost_tsptwd` simule la tournée avec propagation complète — retourne la durée totale (toujours finie)
3. Un **or-opt** post-traitement minimise les temps d'attente en réordonnant les visites

In [ ]:
H = 1440  # minutes par jour


def arc_cost_tsptwd(i, j, departure_time, nodes, scale, perturbations):
    """Coût de l'arc (i→j) évalué à l'heure de départ D_i."""
    xi, yi = nodes[i]['x'], nodes[i]['y']
    xj, yj = nodes[j]['x'], nodes[j]['y']
    base = math.hypot(xi - xj, yi - yj) * scale
    uid, vid = nodes[i]['id'], nodes[j]['id']
    for p in perturbations:
        if [uid, vid] == p['arc'] or [vid, uid] == p['arc']:
            if p['t_start'] <= departure_time <= p['t_end']:
                return base * p['alpha']
    return base


def waiting_time_tw(arrival, node):
    """Temps d'attente avec fenêtre glissante multi-jour.

    - Arrivée avant ouverture : attente jusqu'à a_i
    - Arrivée dans [a_i, b_i] : pas d'attente
    - Arrivée après b_i : attente jusqu'au prochain a_i + k*1440
    """
    a = node.get('a', 0.0) or 0.0
    b = node.get('b')
    if b is None:
        return max(0.0, a - arrival)
    if arrival < a:
        return a - arrival
    if arrival <= b:
        return 0.0
    k = math.ceil((arrival - a) / H)
    return (a + k * H) - arrival


def tour_cost_tsptwd(hamiltonian, data, scale=200):
    """Simule la tournée avec fenêtres glissantes (multi-jour).

    Propagation : D_i = tau_i + w_i + s_i,  tau_{k+1} = D_i + c_ij(D_i)
    Retourne la durée totale (toujours finie).
    """
    nodes = [data['depot']] + data['clients']
    perts = data['perturbations']

    tau = 0.0
    for k in range(len(hamiltonian) - 1):
        i = hamiltonian[k]
        j = hamiltonian[k + 1]
        wait = waiting_time_tw(tau, nodes[i])
        D = tau + wait + nodes[i]['service']
        tau = D + arc_cost_tsptwd(i, j, D, nodes, scale, perts)

    return tau


def or_opt_repair(hamiltonian, data, scale=200):
    """Or-opt : réinsère chaque client à sa meilleure position pour minimiser
    la durée totale (incluant les temps d'attente).
    """
    best = hamiltonian[1:-1]
    best_cost = tour_cost_tsptwd([0] + best + [0], data, scale)

    improved = True
    while improved:
        improved = False
        for i in range(len(best)):
            city = best[i]
            rem = best[:i] + best[i+1:]
            for j in range(len(rem) + 1):
                if j == i:
                    continue
                candidate = rem[:j] + [city] + rem[j:]
                cost = tour_cost_tsptwd([0] + candidate + [0], data, scale)
                if cost < best_cost:
                    best = candidate
                    best_cost = cost
                    improved = True
                    break

    return [0] + best + [0], best_cost


def christofides_tsptwd(G, data, scale=200):
    """Christofides adapté au TSPTW-D avec fenêtres glissantes.

    1. Tour initial via Christofides standard (distances de base)
    2. Or-opt pour minimiser la durée TSPTW-D (attentes + trajets + services)

    Retourne (hamiltonian, cost_tsptwd).
    """
    hamiltonian = christofides(G)
    hamiltonian, cost = or_opt_repair(hamiltonian, data, scale)
    return hamiltonian, cost


# ── Démonstration sur n=10 ────────────────────────────────────────────────────
G10, data10 = generate_graph_from_json("datasets/tsptwd_n10.json")
scale10 = data10['meta']['scale']

ham_base = christofides(G10)
cost_base_only = tour_cost(G10, ham_base, scale10)
cost_base_tsptwd = tour_cost_tsptwd(ham_base, data10, scale10)

print("=== Tour Christofides standard ===")
print(f"  Circuit          : {' → '.join(str(v) for v in ham_base)}")
print(f"  Coût distance    : {cost_base_only:.1f} min")
print(f"  Coût TSPTWD-D    : {cost_base_tsptwd:.1f} min")

ham_tw, cost_tw = christofides_tsptwd(G10, data10, scale10)
print("\n=== Tour Christofides TSPTW-D (après or-opt) ===")
print(f"  Circuit          : {' → '.join(str(v) for v in ham_tw)}")
print(f"  Coût TSPTWD-D    : {cost_tw:.1f} min")

In [ ]:
DATASETS = [
    ("datasets/tsptwd_n10.json",   10),
    ("datasets/tsptwd_n50.json",   50),
    ("datasets/tsptwd_n100.json",  100),
    ("datasets/tsptwd_n200.json",  200),
    ("datasets/tsptwd_n300.json",  300),
]

results_tsptwd = []

for path, n_nodes in DATASETS:
    G_i, data_i = generate_graph_from_json(path)
    scale_i = data_i['meta']['scale']
    t0 = time.perf_counter()
    ham, cost = christofides_tsptwd(G_i, data_i, scale_i)
    elapsed = time.perf_counter() - t0

    cost_dist = tour_cost(G_i, ham, scale_i)

    results_tsptwd.append({
        "n":           n_nodes,
        "coût dist":   round(cost_dist, 1),
        "coût TSPTWD": round(cost, 1),
        "temps (s)":   round(elapsed, 3),
    })
    print(f"n={n_nodes:>4} | dist={cost_dist:.0f} | TSPTWD={cost:.0f} min | {elapsed:.2f}s")

display(pd.DataFrame(results_tsptwd).set_index("n"))

In [ ]:
def benchmark(G, n_nodes, iters=5):
    """Exécute Christofides `iters` fois et retourne des statistiques complètes."""
    costs, times = [], []
    for _ in range(iters):
        t0 = time.perf_counter()
        hamiltonian = christofides(G)
        elapsed = time.perf_counter() - t0
        costs.append(tour_cost(G, hamiltonian))
        times.append(elapsed)
    return {
        "n":           n_nodes,
        "coût moy":    round(np.mean(costs), 1),
        "coût std":    round(np.std(costs), 1),
        "coût min":    round(np.min(costs), 1),
        "coût max":    round(np.max(costs), 1),
        "tps moy (s)": round(np.mean(times), 4),
        "tps std (s)": round(np.std(times), 4),
        "tps min (s)": round(np.min(times), 4),
        "tps max (s)": round(np.max(times), 4),
    }


DATASETS = [
    ("datasets/tsptwd_n10.json",   10),
    ("datasets/tsptwd_n50.json",   50),
    ("datasets/tsptwd_n100.json",  100),
    ("datasets/tsptwd_n200.json",  200),
    ("datasets/tsptwd_n300.json",  300),
    ("datasets/tsptwd_n500.json",  500),
    ("datasets/tsptwd_n1000.json", 1000),
]

graphs = [(generate_graph_from_json(path)[0], n) for path, n in DATASETS]

In [ ]:
results = []
for G_i, n in graphs:
    print(f"  n={n:>5} ...", end=" ", flush=True)
    r = benchmark(G_i, n, iters=5)
    results.append(r)
    print(f"tps={r['tps moy (s)']:.4f}s  coût={r['coût moy']:.1f}")

df = pd.DataFrame(results).set_index("n")
display(df)

In [ ]:
ns         = [r["n"]           for r in results]
times_mean = [r["tps moy (s)"] for r in results]
times_std  = [r["tps std (s)"] for r in results]
costs_mean = [r["coût moy"]    for r in results]
costs_std  = [r["coût std"]    for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Complexité temporelle ────────────────────────────────────────────────────
ax1.errorbar(ns, times_mean, yerr=times_std,
             fmt="o-", color="steelblue", capsize=5, label="Temps moyen ± std")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Nombre de sommets (n)")
ax1.set_ylabel("Temps (s)")
ax1.set_title("Complexité temporelle (log-log)")
ax1.grid(True, which="both", alpha=0.3)

try:
    from scipy.optimize import curve_fit
    def power_law(x, a, b):
        return a * np.array(x) ** b
    popt, _ = curve_fit(power_law, ns, times_mean, p0=[1e-6, 2])
    ns_fit = np.geomspace(min(ns), max(ns), 200)
    ax1.plot(ns_fit, power_law(ns_fit, *popt), "--", color="tomato",
             label=f"Fit : O(n^{popt[1]:.2f})")
except Exception:
    pass
ax1.legend()

# ── Évolution du coût ────────────────────────────────────────────────────────
ax2.errorbar(ns, costs_mean, yerr=costs_std,
             fmt="s-", color="mediumseagreen", capsize=5, label="Coût moyen ± std")
ax2.set_xlabel("Nombre de sommets (n)")
ax2.set_ylabel("Coût du circuit")
ax2.set_title("Évolution du coût selon n")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════
#  TEST RAPIDE — modifier TEST_PATH pour changer d'instance
# ═══════════════════════════════════════════════════════════
TEST_PATH = "datasets/tsptwd_n10.json"

# ── Exécution ────────────────────────────────────────────────
G_t, data_t = generate_graph_from_json(TEST_PATH)
scale_t  = data_t['meta']['scale']
nodes_t  = [data_t['depot']] + data_t['clients']

t0 = time.perf_counter()
ham_t, cost_t = christofides_tsptwd(G_t, data_t, scale_t)
elapsed_t = time.perf_counter() - t0

print(f"Temps d'exécution : {elapsed_t:.4f} s")
print(f"Temps du tour     : {cost_t:.1f} min ({cost_t/60:.1f} h)")
print(f"Circuit           : {' → '.join(str(v) for v in ham_t)}")

# ── Tableau des arrivées ──────────────────────────────────────
def get_arrivals(ham, data, scale):
    """(node_idx, tau_i, wait_i, a_i, b_i, late) pour chaque visite."""
    nodes = [data['depot']] + data['clients']
    perts = data['perturbations']
    tau = 0.0
    rows = []
    for k in range(len(ham) - 1):
        i = ham[k]
        wait = waiting_time_tw(tau, nodes[i])
        a_i  = nodes[i].get('a', 0.0) or 0.0
        b_i  = nodes[i].get('b')
        late = (b_i is not None and tau > b_i)
        rows.append((i, round(tau, 1), round(wait, 1), a_i, b_i, late))
        D   = tau + wait + nodes[i]['service']
        tau = D + arc_cost_tsptwd(i, ham[k+1], D, nodes, scale, perts)
    return rows

arrivals = get_arrivals(ham_t, data_t, scale_t)
print(f"\n  {'Nœud':<6} {'Arrivée':>9} {'Attente':>8}  Fenêtre         Statut")
for node, tau, wait, a, b, late in arrivals:
    b_str  = f"{b:.0f}" if b is not None else "∞"
    status = f"⏳ attente J+{math.ceil((tau-a)/H) if late else ''} ({wait:.0f} min)" if late else ("⏰ attente ouverture" if wait > 0 else "✓")
    print(f"  v{node:<5} {tau:>9.1f} {wait:>8.1f}  [{a:.0f}, {b_str}]  {status}")

# ── Graphe du tour ────────────────────────────────────────────
import matplotlib.patches as mpatches

pos_t = nx.get_node_attributes(G_t, "pos")
late_nodes = {node for node, _, _, _, _, late in arrivals if late}

fig, ax = plt.subplots(figsize=(9, 7))

nx.draw_networkx_edges(G_t, pos_t, edge_color='#e0e0e0', width=0.6, ax=ax)

tour_edges     = list(zip(ham_t[:-1], ham_t[1:]))
normal_edges   = []
perturbed_edges = []
for u, v in tour_edges:
    uid, vid = nodes_t[u]['id'], nodes_t[v]['id']
    if any([uid, vid] == p['arc'] or [vid, uid] == p['arc'] for p in data_t['perturbations']):
        perturbed_edges.append((u, v))
    else:
        normal_edges.append((u, v))

nx.draw_networkx_edges(G_t, pos_t, edgelist=normal_edges,    edge_color='#2a9d8f', width=2.5, ax=ax)
nx.draw_networkx_edges(G_t, pos_t, edgelist=perturbed_edges, edge_color='#e63946', width=2.5, ax=ax)

node_colors = []
for i in range(len(nodes_t)):
    if i == 0:
        node_colors.append('#f4a261')   # dépôt
    elif i in late_nodes:
        node_colors.append('#e9c46a')   # arrivée tardive (attente J+)
    else:
        node_colors.append('#457b9d')   # ok

nx.draw_networkx_nodes(G_t, pos_t, node_color=node_colors, node_size=700, ax=ax)

labels_t = {0: 'Dépôt'}
labels_t.update({
    i: f"v{nodes_t[i]['id']}\n[{nodes_t[i]['a']:.0f},{nodes_t[i]['b']:.0f}]"
    for i in range(1, len(nodes_t))
})
nx.draw_networkx_labels(G_t, pos_t, labels=labels_t, font_size=7, font_color='white', ax=ax)

edge_labels_t = {(ham_t[k], ham_t[k+1]): str(k+1) for k in range(len(ham_t)-1)}
nx.draw_networkx_edge_labels(G_t, pos_t, edge_labels=edge_labels_t, font_size=8, ax=ax)

ax.legend(handles=[
    mpatches.Patch(color='#f4a261', label='Dépôt'),
    mpatches.Patch(color='#457b9d', label='Client (arrivée dans fenêtre)'),
    mpatches.Patch(color='#e9c46a', label='Client (arrivée tardive, attente J+)'),
    mpatches.Patch(color='#2a9d8f', label='Arc du tour'),
    mpatches.Patch(color='#e63946', label='Arc perturbé'),
], loc='lower right', fontsize=8)

n_late = len(late_nodes)
ax.set_title(
    f"Christofides TSPTW-D — n={len(nodes_t)-1} clients\n"
    f"Durée : {cost_t:.1f} min ({cost_t/60:.1f} h)  |  Exec : {elapsed_t:.4f} s  |  {n_late} arrivée(s) tardive(s)",
    fontsize=11
)
ax.axis('off')
plt.tight_layout()
plt.show()